In [1]:
# 1. Mount Google Drive to access the uploaded zip file and save outputs
from google.colab import drive
drive.mount('/content/drive')

# 2. Unzip the dataset into Colab's fast local storage environment (/content)
!unzip -q '/content/drive/MyDrive/ENDO_Project/Polyp_Strict_Dataset.zip' -d '/content/dataset_strict'

# 3. Install the official Ultralytics YOLO package for training and inference
!pip install ultralytics

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.4/65.4 kB 6.6 MB/s eta 0:00:00


In [2]:
import yaml

# Correct the base path to include the nested folder created during unzipping
dataset_config = {
    'path': '/content/dataset_strict/YOLO_Dataset_Strict',
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 1,
    'names': {0: 'polyp'}
}

# Overwrite the existing configuration file with the updated paths
with open('/content/dataset.yaml', 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print("dataset.yaml has been successfully updated with the correct nested path.")

dataset.yaml has been successfully updated with the correct nested path.


In [1]:
from ultralytics import YOLO
import shutil

# Initialize the medium model balancing capacity and generalization
model = YOLO('yolo11m.pt')

# Execute the Colab-optimized training pipeline
results = model.train(
    data='/content/dataset.yaml',

    # 1. Colab-Safe Schedule
    epochs=70,
    patience=15,

    # 2. Free Tier Hardware Optimizations
    imgsz=640,
    batch=16,
    device=0,
    workers=2,         # Matched to Colab free tier CPU cores
    cache=False,       # Disabled to prevent system RAM crashing

    # 3. Local Saving to eliminate Google Drive I/O bottleneck
    project='/content/runs',
    name='polyp_detection_fast',
    save_period=10,

    # 4. Medical-Specific Augmentations
    close_mosaic=10,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    mixup=0.0,
    degrees=15.0,
    fliplr=0.5,
    flipud=0.5,

    # 5. Reproducibility
    deterministic=True
)

print("Optimized training completed. Copying results to Google Drive...")

# Automatically copy the best weights and results to Google Drive securely after training
shutil.copytree('/content/runs/polyp_detection_fast', '/content/drive/MyDrive/ENDO_Project/final_results', dirs_exist_ok=True)
print("Data successfully backed up to Google Drive!")

Ultralytics 8.4.129 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=70, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=polyp_detection_fast, nbs=64, nms=Fals

In [2]:
from ultralytics import YOLO

# 1. Load the golden weights (best.pt) saved during the training phase
model = YOLO('/content/runs/polyp_detection_fast/weights/best.pt')

print("Starting official evaluation on the unseen Test Set...")

# 2. Execute validation strictly on the 'test' split defined in dataset.yaml
metrics = model.val(
    data='/content/dataset.yaml',
    split='test',  # Crucial: This forces evaluation on unseen data

    # Save professional evaluation plots (Confusion Matrix, PR Curves)
    project='/content/runs',
    name='official_test_evaluation',

    # Hardware and processing configurations
    device=0,
    batch=16,

    # Generate detailed visual and JSON reports
    plots=True,
    save_json=True
)

# 3. Output the final official metrics to the terminal
print("\n" + "="*40)
print("OFFICIAL TEST SET RESULTS")
print("="*40)
print(f"Mean Average Precision (mAP@50):    {metrics.box.map50:.4f}")
print(f"Mean Average Precision (mAP@50-95): {metrics.box.map:.4f}")
print("="*40)
print("Plots and visual reports saved successfully to: /content/runs/official_test_evaluation")

Starting official evaluation on the unseen Test Set...
Ultralytics 8.4.129 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11m summary (fused): 126 layers, 20,030,803 parameters, 0 gradients, 67.8 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.1±0.0 ms, read: 26.0±11.1 MB/s, size: 117.7 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/dataset_strict/YOLO_Dataset_Strict/labels/test... 740 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 740/740 339.0it/s 2.2s
val: New cache created: /content/dataset_strict/YOLO_Dataset_Strict/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 47/47 2.0it/s 23.7s
                   all        740        802      0.937      0.878      0.941      0.683
Speed: 1.7ms preprocess, 22.6ms inference, 0.0ms loss, 1.4ms postprocess per image
S